# Invoice Payment Delay & Cash-Flow Intelligence Platform

## Week 1 — Data Loading, Cleaning and Preparation

**Selected dataset:** Customer Invoices  
**Objective:** Prepare a reproducible, documented invoice dataset for exploratory analysis and later payment-delay modelling.

This notebook starts from the raw CSV and recreates all Week 1 outputs. Run cells from top to bottom using **Run All**.

## 1. Imports and paths

The path resolver supports running this notebook from either `notebooks/` or the previously used nested `notebooks/notebooks/` folder.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

possible_data_paths = [
    Path("../data/raw/_compare/customer_invoices/dataset.csv"),
    Path("../../data/raw/_compare/customer_invoices/dataset.csv"),
    Path("data/raw/_compare/customer_invoices/dataset.csv"),
]
DATA_PATH = next((p for p in possible_data_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find customer_invoices/dataset.csv. "
        "Check that the raw dataset is in data/raw/_compare/customer_invoices/ "
        "and that this notebook is inside the project."
    )

# Save outputs to the project's data/processed directory.
project_root = DATA_PATH.parents[4]
OUTPUT_DIR = project_root / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("Input dataset:", DATA_PATH.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Working directory: c:\Users\abhin\OneDrive\Documents\invoice-cashflow\notebooks
Input dataset: C:\Users\abhin\OneDrive\Documents\invoice-cashflow\data\raw\_compare\customer_invoices\dataset.csv
Output directory: C:\Users\abhin\OneDrive\Documents\invoice-cashflow\data\processed


## 2. Load and inspect the raw dataset

In [2]:
raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()

print("Original shape:", df.shape)
display(df.head())
display(pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}))

Original shape: (50000, 19)


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,0200769623,WAL-MAR corp,2020-02-11 00:00:00,2020.0,1.930438e+09,2020-01-26,20200125,20200126,20200210.0,USD,RV,1.0,NaN,54273.28,20200126.0,NAH4,1.930438e+09,0
1,U001,0200980828,BEN E,2019-08-08 00:00:00,2019.0,1.929646e+09,2019-07-22,20190722,20190722,20190811.0,USD,RV,1.0,NaN,79656.60,20190722.0,NAD1,1.929646e+09,0
2,U001,0200792734,MDV/ trust,2019-12-30 00:00:00,2019.0,1.929874e+09,2019-09-14,20190914,20190914,20190929.0,USD,RV,1.0,NaN,2253.86,20190914.0,NAA8,1.929874e+09,0
3,CA02,0140105686,SYSC llc,NaN,2020.0,2.960623e+09,2020-03-30,20200330,20200330,20200410.0,CAD,RV,1.0,NaN,3299.70,20200331.0,CA10,2.960623e+09,1
4,U001,0200769623,WAL-MAR foundation,2019-11-25 00:00:00,2019.0,1.930148e+09,2019-11-13,20191113,20191113,20191128.0,USD,RV,1.0,NaN,33133.29,20191113.0,NAH4,1.930148e+09,0


,data_type,missing_count,missing_percent
business_code,str,0,0.00
cust_number,str,0,0.00
name_customer,str,0,0.00
clear_date,str,10000,20.00
buisness_year,float64,0,0.00
doc_id,float64,0,0.00
posting_date,str,0,0.00
document_create_date,int64,0,0.00
document_create_date.1,int64,0,0.00
due_in_date,float64,0,0.00


## 3. Remove the fully empty column

`area_business` contains no values in the source data, so it contributes no information and is removed. Missing `invoice_id` values are not treated as grounds for dropping otherwise valid invoices.

In [3]:
empty_columns = [col for col in df.columns if df[col].isna().all()]
print("Fully empty columns:", empty_columns)
df = df.drop(columns=empty_columns)
print("Shape after empty-column removal:", df.shape)

Fully empty columns: ['area_business']
Shape after empty-column removal: (50000, 18)


## 4. Remove exact duplicate records

Only fully identical rows are removed. This avoids deleting records merely because an identifier field is missing or repeated.

In [4]:
rows_before_dedup = len(df)
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Rows before:", rows_before_dedup)
print("Exact duplicates removed:", duplicate_count)
print("Rows after:", len(df))
print("Remaining exact duplicates:", int(df.duplicated().sum()))
print("Unique document IDs:", df["doc_id"].nunique())

Rows before: 50000
Exact duplicates removed: 1161
Rows after: 48839
Remaining exact duplicates: 0
Unique document IDs: 48839


## 5. Parse date columns

The source stores `clear_date` and `posting_date` as date strings, while the other four date fields are numeric `YYYYMMDD` values. These formats are parsed separately to avoid interpreting numeric dates as nanoseconds after the Unix epoch.

In [5]:
string_date_columns = ["clear_date", "posting_date"]
numeric_date_columns = [
    "document_create_date",
    "document_create_date.1",
    "due_in_date",
    "baseline_create_date",
]

for col in string_date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in numeric_date_columns:
    df[col] = pd.to_datetime(
        df[col].astype("Int64").astype(str),
        format="%Y%m%d",
        errors="coerce",
    )

date_columns = string_date_columns + numeric_date_columns
print("Date column types:")
display(df[date_columns].dtypes.to_frame("data_type"))
print("Missing dates:")
display(df[date_columns].isna().sum().to_frame("missing_count"))
print("Date ranges:")
display(df[date_columns].agg(["min", "max"]))

Date column types:


,data_type
clear_date,datetime64[us]
posting_date,datetime64[us]
document_create_date,datetime64[us]
document_create_date.1,datetime64[us]
due_in_date,datetime64[us]
baseline_create_date,datetime64[us]


Missing dates:


,missing_count
clear_date,9681
posting_date,0
document_create_date,0
document_create_date.1,0
due_in_date,0
baseline_create_date,0


Date ranges:


,clear_date,posting_date,document_create_date,document_create_date.1,due_in_date,baseline_create_date
min,2019-01-03,2018-12-30,2018-12-26,2018-12-30,2018-12-24,2018-12-14
max,2020-05-22,2020-05-22,2020-05-22,2020-05-22,2020-07-10,2020-05-22


## 6. Create payment outcome fields

Definitions used in this notebook:

- `days_to_pay`: calendar days from **posting date** to actual clearing date.
- `delay`: calendar days from due date to actual clearing date; positive means late, zero means on due date, negative means early.
- `paid_late`: 1 for late, 0 for paid on time/early, and missing for invoices without a clearing date.
- `open_paid_flag`: 1 for paid and 0 for open.

The posting-date anchor is the current project assumption and should be confirmed with the mentor if the business definition expects a different invoice-date field.

In [6]:
paid_mask = df["clear_date"].notna()

df["days_to_pay"] = np.nan
df.loc[paid_mask, "days_to_pay"] = (
    df.loc[paid_mask, "clear_date"] - df.loc[paid_mask, "posting_date"]
).dt.days

df["delay"] = np.nan
df.loc[paid_mask, "delay"] = (
    df.loc[paid_mask, "clear_date"] - df.loc[paid_mask, "due_in_date"]
).dt.days

df["paid_late"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
df.loc[paid_mask, "paid_late"] = (df.loc[paid_mask, "delay"] > 0).astype("Int64")
df["open_paid_flag"] = paid_mask.astype(int)

display(df[[
    "posting_date", "due_in_date", "clear_date",
    "days_to_pay", "delay", "paid_late", "open_paid_flag"
]].head(10))

,posting_date,due_in_date,clear_date,days_to_pay,delay,paid_late,open_paid_flag
0,2020-01-26,2020-02-10,2020-02-11,16.0,1.0,1,1
1,2019-07-22,2019-08-11,2019-08-08,17.0,-3.0,0,1
2,2019-09-14,2019-09-29,2019-12-30,107.0,92.0,1,1
3,2020-03-30,2020-04-10,NaT,NaN,NaN,<NA>,0
4,2019-11-13,2019-11-28,2019-11-25,12.0,-3.0,0,1
5,2019-09-20,2019-10-04,2019-12-04,75.0,61.0,1,1
6,2019-11-01,2019-11-16,2019-11-12,11.0,-4.0,0,1
7,2020-03-19,2020-04-03,NaT,NaN,NaN,<NA>,0
8,2019-06-07,2019-06-22,2019-06-18,11.0,-4.0,0,1
9,2019-02-20,2019-03-07,2019-03-06,14.0,-1.0,0,1


## 7. Date relationship checks and anomaly flags

Date relationships are checked, not silently corrected. Records with unusual date ordering are retained and flagged for investigation.

In [7]:
date_checks = {
    "due_before_posting": df["due_in_date"] < df["posting_date"],
    "clear_before_posting": df["clear_date"] < df["posting_date"],
    "clear_before_baseline": df["clear_date"] < df["baseline_create_date"],
    "posting_before_document": df["posting_date"] < df["document_create_date"],
}

anomaly_counts = {name: int(condition.sum()) for name, condition in date_checks.items()}
display(pd.DataFrame.from_dict(anomaly_counts, orient="index", columns=["affected_rows"]))

df["flag_due_before_posting"] = date_checks["due_before_posting"].astype(int)
df["flag_clear_before_baseline"] = date_checks["clear_before_baseline"].astype(int)
df["flag_posting_before_document"] = date_checks["posting_before_document"].astype(int)

print("Anomaly flags created:")
display(df[[
    "flag_due_before_posting",
    "flag_clear_before_baseline",
    "flag_posting_before_document",
]].sum().to_frame("flagged_rows"))

,affected_rows
due_before_posting,137
clear_before_posting,0
clear_before_baseline,2
posting_before_document,3526


Anomaly flags created:


,flagged_rows
flag_due_before_posting,137
flag_clear_before_baseline,2
flag_posting_before_document,3526


## 8. Target and status summary

In [8]:
print("Paid/open counts (1 = paid, 0 = open):")
display(df["open_paid_flag"].value_counts().sort_index())

print("Paid-late distribution (NA = open invoice):")
display(df["paid_late"].value_counts(dropna=False))

print("Payment duration and delay statistics:")
display(df[["days_to_pay", "delay"]].describe())

Paid/open counts (1 = paid, 0 = open):


open_paid_flag
0     9681
1    39158
Name: count, dtype: int64

Paid-late distribution (NA = open invoice):


paid_late
0       22740
1       16418
<NA>     9681
Name: count, dtype: Int64

Payment duration and delay statistics:


,days_to_pay,delay
count,39158.000000,39158.000000
mean,18.064993,0.837147
std,13.357337,10.843083
min,0.000000,-89.000000
25%,12.000000,-3.000000
50%,15.000000,0.000000
75%,17.000000,2.000000
max,205.000000,204.000000


## 9. Data dictionary

The mapping column connects available source fields to the project's invoice concepts. A mapping of `Not available` means the source does not provide a separate field for that concept. `clear_date` is historical outcome information, not a predictor available at invoice posting time.

In [9]:
descriptions = {
    "business_code": "Business entity code",
    "cust_number": "Customer identifier",
    "name_customer": "Customer name",
    "clear_date": "Actual payment/clearing date",
    "buisness_year": "Business year (source spelling retained)",
    "doc_id": "Document identifier",
    "posting_date": "Invoice posting date",
    "document_create_date": "Document creation date",
    "document_create_date.1": "Additional document creation date field",
    "due_in_date": "Invoice due date",
    "invoice_currency": "Invoice currency",
    "document type": "Document type",
    "posting_id": "Posting identifier",
    "total_open_amount": "Invoice amount/open amount field as supplied by source",
    "baseline_create_date": "Baseline date",
    "cust_payment_terms": "Customer payment terms",
    "invoice_id": "Invoice identifier; six values are missing",
    "isOpen": "Original status: 1 = open, 0 = paid",
    "days_to_pay": "Days between posting date and clearing date",
    "delay": "Days between due date and clearing date",
    "paid_late": "1 = paid late, 0 = paid on time/early, NA = unpaid",
    "open_paid_flag": "1 = paid, 0 = open",
    "flag_due_before_posting": "1 if due date precedes posting date",
    "flag_clear_before_baseline": "1 if clearing date precedes baseline date",
    "flag_posting_before_document": "1 if posting date precedes document creation date",
}

project_mapping = {
    "doc_id": "Invoice/document ID",
    "invoice_id": "Invoice ID",
    "cust_number": "Customer ID",
    "name_customer": "Customer name",
    "total_open_amount": "Invoice amount",
    "posting_date": "Invoice posting date (chosen date anchor)",
    "due_in_date": "Due date",
    "clear_date": "Actual payment date (historical outcome)",
    "cust_payment_terms": "Payment terms",
    "invoice_currency": "Currency",
    "days_to_pay": "Payment duration target",
    "delay": "Payment delay target",
    "paid_late": "Late-payment classification target",
    "open_paid_flag": "Paid/open status target",
}

data_dictionary = pd.DataFrame({
    "column_name": df.columns,
    "data_type": [str(df[col].dtype) for col in df.columns],
    "description": [descriptions[col] for col in df.columns],
    "project_mapping": [project_mapping.get(col, "Supporting attribute / not separately mapped") for col in df.columns],
})

display(data_dictionary.head(30))

,column_name,data_type,description,project_mapping
0,business_code,str,Business entity code,Supporting attribute / not separately mapped
1,cust_number,str,Customer identifier,Customer ID
2,name_customer,str,Customer name,Customer name
3,clear_date,datetime64[us],Actual payment/clearing date,Actual payment date (historical outcome)
4,buisness_year,float64,Business year (source spelling retained),Supporting attribute / not separately mapped
5,doc_id,float64,Document identifier,Invoice/document ID
6,posting_date,datetime64[us],Invoice posting date,Invoice posting date (chosen date anchor)
7,document_create_date,datetime64[us],Document creation date,Supporting attribute / not separately mapped
8,document_create_date.1,datetime64[us],Additional document creation date field,Supporting attribute / not separately mapped
9,due_in_date,datetime64[us],Invoice due date,Due date


## 10. Cleaning log

The log records the issue/action/reason and the number of affected rows. Date anomalies are retained and flagged rather than dropped.

In [10]:
cleaning_log = pd.DataFrame([
    ["Fully empty column", "Removed area_business", "Column contained no usable values", len(raw_df)],
    ["Exact duplicate records", f"Removed {duplicate_count} exact duplicate rows", "Avoid duplicate counting while retaining non-identical records", duplicate_count],
    ["Missing clear_date", "Retained missing values for open invoices", "No actual payment date exists yet; do not invent an outcome", int(df["clear_date"].isna().sum())],
    ["Missing invoice_id", "Retained records with missing invoice_id", "Rows remain otherwise valid and have document identifiers", int(df["invoice_id"].isna().sum())],
    ["Date conversion", "Parsed six source date columns into datetime", "Ensure correct date arithmetic and date validation", len(date_columns)],
    ["Payment targets", "Created days_to_pay, delay, paid_late and open_paid_flag", "Represent historical payment outcomes; open invoices remain unlabeled for paid_late", 4],
    ["Date anomalies", "Retained unusual date relationships and created flags", "Preserve source data while making anomalies reviewable", sum(anomaly_counts.values())],
], columns=["issue", "action_taken", "reason", "rows_affected_or_fields"])

display(cleaning_log)

,issue,action_taken,reason,rows_affected_or_fields
0,Fully empty column,Removed area_business,Column contained no usable values,50000
1,Exact duplicate records,Removed 1161 exact duplicate rows,Avoid duplicate counting while retaining non-i...,1161
2,Missing clear_date,Retained missing values for open invoices,No actual payment date exists yet; do not inve...,9681
3,Missing invoice_id,Retained records with missing invoice_id,Rows remain otherwise valid and have document ...,6
4,Date conversion,Parsed six source date columns into datetime,Ensure correct date arithmetic and date valida...,6
5,Payment targets,"Created days_to_pay, delay, paid_late and open...",Represent historical payment outcomes; open in...,4
6,Date anomalies,Retained unusual date relationships and create...,Preserve source data while making anomalies re...,3665


## 11. Export Week 1 deliverables

In [11]:
cleaned_path = OUTPUT_DIR / "customer_invoices_cleaned.csv"
dictionary_path = OUTPUT_DIR / "data_dictionary.csv"
log_path = OUTPUT_DIR / "cleaning_log.csv"

df.to_csv(cleaned_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)
cleaning_log.to_csv(log_path, index=False)

print("Exported deliverables:")
for path in [cleaned_path, dictionary_path, log_path]:
    print(f"- {path.name}: {path.resolve()}")

Exported deliverables:
- customer_invoices_cleaned.csv: C:\Users\abhin\OneDrive\Documents\invoice-cashflow\data\processed\customer_invoices_cleaned.csv
- data_dictionary.csv: C:\Users\abhin\OneDrive\Documents\invoice-cashflow\data\processed\data_dictionary.csv
- cleaning_log.csv: C:\Users\abhin\OneDrive\Documents\invoice-cashflow\data\processed\cleaning_log.csv


## 12. Final verification

In [12]:
cleaned_check = pd.read_csv(cleaned_path)
dictionary_check = pd.read_csv(dictionary_path)
log_check = pd.read_csv(log_path)

verification = {
    "cleaned_dataset_exists": cleaned_path.exists(),
    "data_dictionary_exists": dictionary_path.exists(),
    "cleaning_log_exists": log_path.exists(),
    "cleaned_rows": len(cleaned_check),
    "cleaned_columns": len(cleaned_check.columns),
    "duplicate_rows": int(cleaned_check.duplicated().sum()),
    "missing_clear_date": int(cleaned_check["clear_date"].isna().sum()),
    "missing_invoice_id": int(cleaned_check["invoice_id"].isna().sum()),
    "dictionary_rows": len(dictionary_check),
    "cleaning_log_entries": len(log_check),
}
display(pd.Series(verification, name="result").to_frame())

assert cleaned_check.shape == (48839, 25), "Unexpected cleaned dataset shape; investigate before submission."
assert cleaned_check.duplicated().sum() == 0, "Duplicate rows remain."
assert len(dictionary_check) == len(cleaned_check.columns), "Dictionary does not cover every cleaned column."
print("Week 1 export verification passed.")

,result
cleaned_dataset_exists,True
data_dictionary_exists,True
cleaning_log_exists,True
cleaned_rows,48839
cleaned_columns,25
duplicate_rows,0
missing_clear_date,9681
missing_invoice_id,6
dictionary_rows,25
cleaning_log_entries,7


Week 1 export verification passed.


## 13. Leakage and modelling notes

- Do **not** use `clear_date`, `days_to_pay`, `delay`, or `paid_late` as input features when predicting payment outcomes at invoice creation/posting time. They contain future outcome information.
- `open_paid_flag` also describes the observed outcome/status and must not be used as a predictor for the same task.
- Identifier columns such as `doc_id` and `invoice_id` should not be treated as continuous numerical features.
- The source dataset does not provide a distinct expected payment date or future cash-collection schedule. Cash-flow forecasting will require a clearly documented modelling approach using available invoice dates, amounts, payment terms and predictions.
- The date anchor for `days_to_pay` is currently `posting_date`; confirm this business definition with the mentor before modelling.

## Week 1 completion checkpoint

The notebook creates the cleaned CSV, data dictionary and cleaning log from the raw source. Review the flagged date anomalies and confirm target definitions with the mentor before moving to exploratory data analysis.